# Projekt 4 — Optymalizacja portfela akcji GPW (model Markowitza)
## Zarządzanie Ryzykiem w Przedsiębiorstwie

**Kontekst.** Spółka **XTB S.A.** otrzymała **1 mln PLN** do ulokowania wyłącznie w akcjach
notowanych na **GPW**. Naszym zadaniem jest dobranie portfela, wyznaczenie miar ryzyka
oraz porównanie wyniku z prognozą modelu jednoskaźnikowego (CAPM-like).

---

## Wybór 5 polskich spółek z GPW i uzasadnienie

Cel doboru: **dywersyfikacja sektorowa**, **wysoka płynność** (blue chipy WIG20),
**stabilność** (brak ryzyka delistingu) i **reprezentatywność polskiej gospodarki**.

| # | Ticker | Spółka | Sektor | Uzasadnienie |
|---|--------|--------|--------|--------------|
| 1 | **PKO.WA** | PKO Bank Polski | Banki | Największy polski bank, wysokie dywidendy, β ≈ rynek, ekspozycja na stopę proc. NBP |
| 2 | **PKN.WA** | PKN Orlen | Energetyka / paliwa | Państwowy gigant, hedge na inflację surowcową, wysokie obroty |
| 3 | **KGH.WA** | KGHM Polska Miedź | Surowce (miedź, srebro) | Globalna ekspozycja USD/CNY, dekorelacja z bankami, wysoka zmienność |
| 4 | **CDR.WA** | CD Projekt | Technologie / gaming | Sektor IT, globalne przychody w EUR/USD, niska korelacja z finansami |
| 5 | **DNP.WA** | Dino Polska | Consumer staples (retail spożywczy) | Defensywny, niska β, stabilny cashflow — balans dla portfela |

**Dlaczego taki dobór?** Otrzymujemy ekspozycję na **5 niezależnych sektorów**
(banki, energia, surowce, IT, retail), co maksymalizuje korzyść z dywersyfikacji.
Wszystkie spółki są w **WIG20** — najwyższa płynność na GPW, łatwe otwarcie i zamknięcie
pozycji za 1 mln PLN bez znaczącego *slippage*.

**Stopa wolna od ryzyka.** Przyjmujemy rentowność polskich krótkoterminowych obligacji
skarbowych (bony 52-tygodniowe) na poziomie **r_f ≈ 5,25 % rocznie** (stan przybliżony
po cyklu obniżek RPP w 2025 r.).

**Indeks rynku.** **WIG20** — benchmark dużych spółek GPW (ticker `WIG20.WA` w Yahoo Finance).

W pkt 1 (portfel dwuskładnikowy) bierzemy **PKO.WA + CDR.WA** — kombinację banki + tech,
która empirycznie wykazuje **niską korelację** i dobrze ilustruje efekt dywersyfikacji.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

# --- Parametry projektu ---
TICKERS_5 = ['PKO.WA', 'PKN.WA', 'KGH.WA', 'CDR.WA', 'DNP.WA']
NAMES = {
    'PKO.WA': 'PKO BP',
    'PKN.WA': 'PKN Orlen',
    'KGH.WA': 'KGHM',
    'CDR.WA': 'CD Projekt',
    'DNP.WA': 'Dino Polska',
}
INDEX = 'WIG20.WA'

START   = '2021-01-01'
END     = '2026-05-13'
RF_ANN  = 0.0525           # 5,25 % rocznie (krótkoterminowe obligacje PL)
TRADING_DAYS = 252
RF_D    = RF_ANN / TRADING_DAYS

CAPITAL_PLN = 1_000_000    # 1 mln PLN do ulokowania

print(f'Spółki: {", ".join(NAMES.values())}')
print(f'Indeks: {INDEX}')
print(f'Okres:  {START} — {END}')
print(f'r_f rocznie: {RF_ANN:.2%}   r_f dziennie: {RF_D:.6f}')

In [ ]:
# --- Pobranie danych z Yahoo Finance ---
all_tickers = TICKERS_5 + [INDEX]
raw = yf.download(all_tickers, start=START, end=END, progress=False, auto_adjust=False)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Adj Close'].copy() if 'Adj Close' in raw.columns.levels[0] else raw['Close'].copy()
else:
    prices = raw[['Adj Close']].copy()

prices = prices[all_tickers].dropna()
prices.columns = [NAMES.get(c, c) for c in prices.columns]

# log-zwroty
log_ret = np.log(prices / prices.shift(1)).dropna()

# Indeks osobno
ret_index = log_ret[INDEX if INDEX in log_ret.columns else log_ret.columns[-1]]
log_ret_assets = log_ret[[NAMES[t] for t in TICKERS_5]]

print(f'Obserwacje:  {len(log_ret_assets)} dni roboczych')
print(f'Zakres:      {log_ret_assets.index[0].date()} — {log_ret_assets.index[-1].date()}')
display(prices.tail())

In [ ]:
# --- Wizualizacja: ceny i log-zwroty ---
fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Ceny znormalizowane do 100
norm_prices = prices[[NAMES[t] for t in TICKERS_5]] / prices[[NAMES[t] for t in TICKERS_5]].iloc[0] * 100
norm_prices.plot(ax=axes[0], lw=1.2)
axes[0].set_title('Ceny znormalizowane do 100 (start okresu)', fontweight='bold')
axes[0].set_ylabel('Indeks (start = 100)')
axes[0].grid(alpha=0.3); axes[0].legend(loc='upper left')

# Log-zwroty
log_ret_assets.plot(ax=axes[1], lw=0.5, alpha=0.7)
axes[1].set_title('Logarytmiczne stopy zwrotu (dzienne)', fontweight='bold')
axes[1].set_ylabel('log-return')
axes[1].axhline(0, color='black', lw=0.5); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
# 1. Portfel **dwuskładnikowy** — PKO BP & CD Projekt

Wybieramy parę z różnych sektorów (banki vs tech) w celu pokazania efektu dywersyfikacji.

### Notacja
- $R_p = w_1 R_1 + w_2 R_2,\quad w_1 + w_2 = 1$
- $\mathbb{E}(R_p) = w_1 \mu_1 + w_2 \mu_2$
- $\sigma_p^2 = w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2 + 2 w_1 w_2 \rho_{12}\sigma_1\sigma_2$

In [ ]:
# --- 1. Statystyki dla pary PKO BP & CD Projekt ---
pair = ['PKO BP', 'CD Projekt']
R = log_ret_assets[pair]

mu  = R.mean() * TRADING_DAYS          # roczne
sig = R.std()  * np.sqrt(TRADING_DAYS) # roczne
rho = R.corr().iloc[0, 1]

print('--- Statystyki roczne ---')
print(f'  E(R_{pair[0]}) = {mu.iloc[0]:.4f}   σ = {sig.iloc[0]:.4f}')
print(f'  E(R_{pair[1]}) = {mu.iloc[1]:.4f}   σ = {sig.iloc[1]:.4f}')
print(f'  Korelacja ρ_{{12}} = {rho:.4f}')

Sigma2 = R.cov().values * TRADING_DAYS  # macierz kowariancji roczna
mu_v   = mu.values
print('\nMacierz kowariancji (roczna):')
display(pd.DataFrame(Sigma2, index=pair, columns=pair).round(6))

## 1a + 1b — Stopy zwrotu i ryzyko portfela dla różnych wag

Rysujemy zbiór możliwych portfeli $(\sigma_p, \mathbb{E}(R_p))$.
Zaznaczamy dodatkowo dwa konkretne portfele:
- **wagi równe** $w = (0.5,\,0.5)$
- **wagi proporcjonalne do kapitalizacji** (przybliżone — patrz komórka).

In [ ]:
# --- 1a/1b: krzywa możliwych portfeli ---
w_grid = np.linspace(0, 1, 201)
mu_p  = np.array([w*mu_v[0] + (1-w)*mu_v[1] for w in w_grid])
sig_p = np.array([np.sqrt(w**2*Sigma2[0,0] + (1-w)**2*Sigma2[1,1]
                          + 2*w*(1-w)*Sigma2[0,1]) for w in w_grid])

# Portfel równowagowy
w_eq = np.array([0.5, 0.5])
mu_eq  = w_eq @ mu_v
sig_eq = np.sqrt(w_eq @ Sigma2 @ w_eq)

# Wagi proporcjonalne do kapitalizacji rynkowej (orientacyjne — maj 2025)
#   PKO BP  ≈  75 mld PLN
#   CD PROJEKT ≈ 16 mld PLN
cap = np.array([75.0, 16.0])
w_cap = cap / cap.sum()
mu_cap  = w_cap @ mu_v
sig_cap = np.sqrt(w_cap @ Sigma2 @ w_cap)

print(f'Portfel równowagowy (50/50):           E(R)={mu_eq:.4f}, σ={sig_eq:.4f}')
print(f'Portfel proporcjonalny do kapitaliz.:  w={w_cap.round(3)}  →  E(R)={mu_cap:.4f}, σ={sig_cap:.4f}')

## 1c — Portfel o **minimalnym ryzyku** (MVP)

Dla dwóch składników (bez krótkiej sprzedaży i przy $w_1+w_2=1$):
$$
w_1^{*} = \frac{\sigma_2^2 - \rho\,\sigma_1\sigma_2}{\sigma_1^2 + \sigma_2^2 - 2\rho\,\sigma_1\sigma_2}
$$

In [ ]:
# --- 1c: MVP analitycznie ---
s1, s2 = sig.iloc[0], sig.iloc[1]
w1_mvp = (s2**2 - rho*s1*s2) / (s1**2 + s2**2 - 2*rho*s1*s2)
w1_mvp = float(np.clip(w1_mvp, 0, 1))
w_mvp  = np.array([w1_mvp, 1 - w1_mvp])
mu_mvp  = w_mvp @ mu_v
sig_mvp = np.sqrt(w_mvp @ Sigma2 @ w_mvp)
print(f'MVP:  w = ({w_mvp[0]:.4f}, {w_mvp[1]:.4f})  →  E(R)={mu_mvp:.4f},  σ={sig_mvp:.4f}')

## 1d — Portfel o minimalnym ryzyku przy **ustalonej stopie zwrotu** $R^\star$

Dla pary składników $R^\star$ daje **jednoznacznie** wagi:
$w_1 = \frac{R^\star - \mu_2}{\mu_1 - \mu_2}$, $w_2 = 1 - w_1$.
Bierzemy $R^\star = 0{,}15$ (15 % rocznie).

In [ ]:
R_target = 0.15
w1_t = (R_target - mu_v[1]) / (mu_v[0] - mu_v[1])
w1_t = float(np.clip(w1_t, 0, 1))
w_target = np.array([w1_t, 1 - w1_t])
mu_t  = w_target @ mu_v
sig_t = np.sqrt(w_target @ Sigma2 @ w_target)
print(f'Cel R*=0.15:  w=({w_target[0]:.4f}, {w_target[1]:.4f})  →  E(R)={mu_t:.4f},  σ={sig_t:.4f}')

## 1e — Portfel **rynkowy** (max Sharpe)

Linia rynku kapitałowego (CML) jest styczna do brzegu efektywnego w portfelu rynkowym.
Maksymalizujemy $S(w) = \dfrac{\mathbb{E}(R_p) - r_f}{\sigma_p}$.

In [ ]:
# --- 1e: portfel rynkowy (max Sharpe) ---
def sharpe(w, mu, Sigma, rf=RF_ANN):
    rp  = w @ mu
    sp  = np.sqrt(w @ Sigma @ w)
    return (rp - rf) / sp

S_grid = np.array([sharpe(np.array([w, 1-w]), mu_v, Sigma2) for w in w_grid])
i_max  = int(np.argmax(S_grid))
w_mkt  = np.array([w_grid[i_max], 1 - w_grid[i_max]])
mu_mkt  = w_mkt @ mu_v
sig_mkt = np.sqrt(w_mkt @ Sigma2 @ w_mkt)
S_mkt   = (mu_mkt - RF_ANN) / sig_mkt
print(f'Portfel rynkowy:  w=({w_mkt[0]:.4f}, {w_mkt[1]:.4f})')
print(f'                  E(R)={mu_mkt:.4f},  σ={sig_mkt:.4f},  Sharpe={S_mkt:.4f}')

In [ ]:
# --- Wykres: zbiór możliwych portfeli + MVP + portfel rynkowy + CML ---
fig, ax = plt.subplots(figsize=(11, 7))

ax.plot(sig_p, mu_p, color='steelblue', lw=2, label='Zbiór możliwych portfeli')
ax.scatter(sig.iloc[0], mu_v[0], s=120, marker='^', color='navy',
           label=f'100% {pair[0]}', zorder=4)
ax.scatter(sig.iloc[1], mu_v[1], s=120, marker='v', color='darkorange',
           label=f'100% {pair[1]}', zorder=4)
ax.scatter(sig_eq, mu_eq,  s=110, color='gray', label='50/50', zorder=4)
ax.scatter(sig_cap, mu_cap, s=110, color='purple', marker='D', label='Wagi ~ kapit.', zorder=4)
ax.scatter(sig_mvp, mu_mvp, s=160, color='green', marker='*',
           label=f'MVP (w={w_mvp[0]:.2f})', zorder=5)
ax.scatter(sig_t, mu_t,    s=130, color='blue',  marker='P',
           label=f'R*=0.15 (w={w_target[0]:.2f})', zorder=5)
ax.scatter(sig_mkt, mu_mkt, s=180, color='red', marker='*',
           label=f'Portfel rynkowy (w={w_mkt[0]:.2f})', zorder=5)

# Linia CML
x_cml = np.linspace(0, sig.max()*1.1, 100)
y_cml = RF_ANN + (mu_mkt - RF_ANN)/sig_mkt * x_cml
ax.plot(x_cml, y_cml, '--', color='red', alpha=0.6, label='CML')

ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5, label=f'r_f={RF_ANN:.2%}')
ax.set_xlabel('Ryzyko σ_p (roczne)'); ax.set_ylabel('Oczekiwana stopa zwrotu E(R_p)')
ax.set_title('Pkt 1: Możliwe portfele 2-składnikowe (PKO BP + CD Projekt)',
             fontweight='bold')
ax.legend(loc='best'); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

---
# 2. VaR portfeli z punktu 1 metodą wariancji-kowariancji

Dla portfela o stopie zwrotu $R_p \sim \mathcal{N}(\mu_p, \sigma_p^2)$ (założenie metody):
$$
\mathrm{VaR}_{\alpha}^{T} = -\big(\mu_p \cdot T + z_\alpha \cdot \sigma_p\cdot\sqrt{T}\big)
$$
gdzie $z_\alpha = \Phi^{-1}(\alpha)$, $\alpha\in\{0{,}05,\,0{,}01\}$.

### Dodatkowe ograniczenie
> *Z prawdopodobieństwem 99% portfel nie może stracić więcej niż 20% wartości w ciągu roku.*

To znaczy: **$\mathrm{VaR}_{99\%}^{1\,\text{rok}} \le 0{,}20$**, czyli
$\mu_p - 2{,}3263\,\sigma_p \ge -0{,}20$.

In [ ]:
# --- 2. VaR (wariancja-kowariancja) dla portfeli z pkt 1 ---
def var_param(mu, sig, alpha, T=1):
    # roczne mu, sig; T w latach
    z = norm.ppf(alpha)
    return -(mu*T + z*sig*np.sqrt(T))

def es_param(mu, sig, alpha, T=1):
    # Expected Shortfall pod normalnością
    return -(mu*T - sig*np.sqrt(T) * norm.pdf(norm.ppf(alpha))/alpha)

def check_constraint_20pct(mu, sig):
    # Czy VaR 99% (roczne) <= 20%?
    return var_param(mu, sig, 0.01, T=1) <= 0.20

ports = {
    'PKO BP (100%)':       (mu_v[0], sig.iloc[0]),
    'CD Projekt (100%)':   (mu_v[1], sig.iloc[1]),
    'Równowagowy 50/50':   (mu_eq,   sig_eq),
    'Proporc. kapit.':     (mu_cap,  sig_cap),
    'MVP':                 (mu_mvp,  sig_mvp),
    f'R*=0.15':            (mu_t,    sig_t),
    'Rynkowy (max Sharpe)':(mu_mkt,  sig_mkt),
}

rows = []
for name, (m, s) in ports.items():
    rows.append({
        'Portfel':       name,
        'E(R) rocznie':  f'{m:.4f}',
        'σ rocznie':     f'{s:.4f}',
        'VaR 95% (rok)': f'{var_param(m, s, 0.05):.4f}',
        'VaR 99% (rok)': f'{var_param(m, s, 0.01):.4f}',
        'ES  99% (rok)': f'{es_param(m, s, 0.01):.4f}',
        'Spełnia ≤20%?': 'TAK ✓' if check_constraint_20pct(m, s) else 'NIE ✗',
    })
var_table = pd.DataFrame(rows)
display(var_table)

## Portfel maksymalizujący $E(R)$ pod ograniczeniem VaR 99% ≤ 20%

Z liniowego ograniczenia $\mu_p - z_{0.01}\sigma_p \ge -0{,}20$ wyznaczamy
brzegowy zbiór dopuszczalnych wag — wzdłuż zbioru możliwych portfeli szukamy maksimum
$E(R_p)$.

In [ ]:
# --- 2. Optymalizacja: max E(R) | VaR_99% <= 0.20 ---
z99 = -norm.ppf(0.01)   # ≈ 2.3263

feas_mask = (mu_p - z99 * sig_p) >= -0.20
if feas_mask.any():
    mu_p_feas = mu_p.copy(); mu_p_feas[~feas_mask] = -np.inf
    i_best = int(np.argmax(mu_p_feas))
    w_best = np.array([w_grid[i_best], 1 - w_grid[i_best]])
    mu_b, sig_b = mu_p[i_best], sig_p[i_best]
    print(f'Maksimum E(R) pod ograniczeniem VaR99% ≤ 20%:')
    print(f'   w = ({w_best[0]:.4f}, {w_best[1]:.4f})')
    print(f'   E(R) = {mu_b:.4f},  σ = {sig_b:.4f}')
    print(f'   VaR 99% (rok) = {var_param(mu_b, sig_b, 0.01):.4f}')
else:
    print('Żaden portfel z pary nie spełnia ograniczenia ≤ 20% strat z prob. 99%.')

# Wizualizacja dopuszczalnego obszaru
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(sig_p, mu_p, color='lightgray', lw=2, label='Zbiór możliwych portfeli')
ax.plot(sig_p[feas_mask], mu_p[feas_mask], color='green', lw=3,
        label='Spełnia VaR99% ≤ 20%')
ax.scatter(sig_mvp, mu_mvp, s=140, color='blue', marker='*', label='MVP')
ax.scatter(sig_mkt, mu_mkt, s=140, color='red', marker='*', label='Portfel rynkowy')
if feas_mask.any():
    ax.scatter(sig_b, mu_b, s=200, color='orange', marker='P',
               label=f'Max E(R) | VaR≤20% (w={w_best[0]:.2f})', zorder=5)
# linia ograniczenia: mu = z99*sig - 0.20
x_c = np.linspace(0, sig_p.max()*1.1, 100)
ax.plot(x_c, z99*x_c - 0.20, '--', color='red', alpha=0.6, label='granica VaR99%=20%')
ax.set_xlabel('σ_p'); ax.set_ylabel('E(R_p)')
ax.set_title('Pkt 2: Portfele dopuszczalne (VaR 99% ≤ 20% strat rocznych)',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

---
# 3. Rozszerzenie portfela do **5 akcji** (optymalizacja numeryczna)

Wektorowy zapis:
$$
\mathbb{E}(R_p) = \mathbf{w}^\top \boldsymbol{\mu},\qquad
\sigma_p^2 = \mathbf{w}^\top \Sigma\, \mathbf{w},\qquad
\sum_i w_i = 1,\ \ w_i \ge 0\ (\text{brak short).}
$$

Wyznaczamy:
- **brzeg efektywny** (Markowitz) — przez sweep po docelowym zwrocie,
- **MVP**, **portfel rynkowy (max Sharpe)** — `scipy.optimize.minimize`,
- **chmurę Monte Carlo** (10 000 losowych portfeli) — tło.

In [ ]:
# --- 3. Portfel 5 akcji: dane wektorowe ---
R5      = log_ret_assets[[NAMES[t] for t in TICKERS_5]]
mu5     = R5.mean().values * TRADING_DAYS
Sigma5  = R5.cov().values * TRADING_DAYS
names5  = list(R5.columns)
n = len(names5)

print('Roczne stopy zwrotu E(R):')
display(pd.Series(mu5, index=names5, name='E(R)').round(4))

print('\nMacierz korelacji:')
display(R5.corr().round(3))

print('\nMacierz kowariancji (roczna):')
display(pd.DataFrame(Sigma5, index=names5, columns=names5).round(5))

In [ ]:
# --- 3. Monte Carlo: chmura 10000 losowych portfeli ---
np.random.seed(42)
N_MC = 10_000
W = np.random.dirichlet(np.ones(n), size=N_MC)   # wagi sumujące się do 1, w_i>=0
mc_mu  = W @ mu5
mc_sig = np.sqrt(np.einsum('ij,jk,ik->i', W, Sigma5, W))
mc_sharpe = (mc_mu - RF_ANN) / mc_sig

In [ ]:
# --- 3. MVP, portfel rynkowy, brzeg efektywny — optymalizacja numeryczna ---
def port_var(w, S): return w @ S @ w
def port_ret(w, m): return w @ m
def neg_sharpe(w, m, S, rf): return -(w @ m - rf) / np.sqrt(w @ S @ w)

bounds = tuple((0.0, 1.0) for _ in range(n))
cons_sum = {'type':'eq', 'fun': lambda w: w.sum() - 1.0}

# MVP
res_mvp = minimize(port_var, x0=np.ones(n)/n, args=(Sigma5,), method='SLSQP',
                   bounds=bounds, constraints=[cons_sum])
w_mvp5  = res_mvp.x
mu_mvp5  = port_ret(w_mvp5, mu5)
sig_mvp5 = np.sqrt(port_var(w_mvp5, Sigma5))

# Portfel rynkowy
res_mkt = minimize(neg_sharpe, x0=np.ones(n)/n, args=(mu5, Sigma5, RF_ANN),
                   method='SLSQP', bounds=bounds, constraints=[cons_sum])
w_mkt5  = res_mkt.x
mu_mkt5  = port_ret(w_mkt5, mu5)
sig_mkt5 = np.sqrt(port_var(w_mkt5, Sigma5))
S_mkt5   = (mu_mkt5 - RF_ANN) / sig_mkt5

# Brzeg efektywny: sweep
mu_targets = np.linspace(mu_mvp5, mu5.max(), 60)
eff_sig = []
eff_mu  = []
for mt in mu_targets:
    cons = [cons_sum, {'type':'eq', 'fun': lambda w, mt=mt: w @ mu5 - mt}]
    res = minimize(port_var, x0=np.ones(n)/n, args=(Sigma5,), method='SLSQP',
                   bounds=bounds, constraints=cons)
    if res.success:
        eff_sig.append(np.sqrt(port_var(res.x, Sigma5)))
        eff_mu.append(mt)
eff_sig = np.array(eff_sig); eff_mu = np.array(eff_mu)

print('--- MVP (5 akcji) ---')
print(pd.Series(w_mvp5, index=names5, name='w_MVP').round(4))
print(f'E(R)={mu_mvp5:.4f}, σ={sig_mvp5:.4f}, Sharpe={(mu_mvp5-RF_ANN)/sig_mvp5:.4f}\n')

print('--- Portfel rynkowy (5 akcji) ---')
print(pd.Series(w_mkt5, index=names5, name='w_MKT').round(4))
print(f'E(R)={mu_mkt5:.4f}, σ={sig_mkt5:.4f}, Sharpe={S_mkt5:.4f}')

In [ ]:
# --- 3. Wykres: brzeg efektywny + chmura MC + porównanie z pkt 1 ---
fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(mc_sig, mc_mu, c=mc_sharpe, cmap='viridis', s=8, alpha=0.5)
plt.colorbar(sc, ax=ax, label='Sharpe ratio')

ax.plot(eff_sig, eff_mu, color='red', lw=2.5, label='Brzeg efektywny (5 akcji)')
ax.plot(sig_p, mu_p, color='gray', lw=2, ls='--', label='Zbiór 2-akcji (pkt 1)')

# Pojedyncze akcje
for i, name in enumerate(names5):
    ax.scatter(np.sqrt(Sigma5[i,i]), mu5[i], s=110, marker='s',
               edgecolor='black', label=name, zorder=4)

ax.scatter(sig_mvp5, mu_mvp5, s=200, color='blue', marker='*',
           label=f'MVP (5)', zorder=5)
ax.scatter(sig_mkt5, mu_mkt5, s=200, color='red',  marker='*',
           label=f'Portfel rynkowy (5)', zorder=5)
ax.scatter(sig_mvp, mu_mvp, s=140, color='lightblue', marker='*',
           edgecolor='navy', label='MVP (2)', zorder=4)
ax.scatter(sig_mkt, mu_mkt, s=140, color='lightcoral', marker='*',
           edgecolor='darkred', label='Rynkowy (2)', zorder=4)

x_cml5 = np.linspace(0, mc_sig.max()*1.05, 100)
ax.plot(x_cml5, RF_ANN + (mu_mkt5-RF_ANN)/sig_mkt5 * x_cml5, '--',
        color='red', alpha=0.5, label='CML (5)')

ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5)
ax.set_xlabel('σ_p (roczne)'); ax.set_ylabel('E(R_p) (roczne)')
ax.set_title('Pkt 3: Brzeg efektywny dla portfela 5 akcji (vs pkt 1)',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=9, ncol=2)
ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

In [ ]:
# --- 3. Porównanie liczbowe MVP/rynkowy: 2 akcje vs 5 akcji ---
rows = [
    ('MVP (2 akcje)',     mu_mvp,  sig_mvp,  (mu_mvp-RF_ANN)/sig_mvp),
    ('MVP (5 akcji)',     mu_mvp5, sig_mvp5, (mu_mvp5-RF_ANN)/sig_mvp5),
    ('Rynkowy (2 akcje)', mu_mkt,  sig_mkt,  (mu_mkt-RF_ANN)/sig_mkt),
    ('Rynkowy (5 akcji)', mu_mkt5, sig_mkt5, S_mkt5),
]
cmp_df = pd.DataFrame(rows, columns=['Portfel','E(R)','σ','Sharpe']).round(4)
display(cmp_df)

# Alokacja kapitału dla portfela rynkowego (5 akcji)
alloc = pd.DataFrame({
    'Waga': np.round(w_mkt5, 4),
    'Kapitał [PLN]': np.round(w_mkt5 * CAPITAL_PLN, 2),
}, index=names5)
print('\nAlokacja 1 000 000 PLN — portfel rynkowy (5 akcji):')
display(alloc)

---
# 4. Model **jednoskaźnikowy** (Sharpe single-index model)

Założenie: zwrot akcji $i$ daje się rozłożyć na ekspozycję na indeks rynkowy $R_M$
i składnik idiosynkratyczny:
$$
R_i = \alpha_i + \beta_i R_M + \varepsilon_i,\qquad
\mathrm{Cov}(\varepsilon_i, \varepsilon_j) = 0\ \text{dla } i\neq j,\quad
\mathrm{Cov}(R_M, \varepsilon_i) = 0.
$$

Stąd macierz kowariancji **z modelu**:
$$
\Sigma_{ij}^{(SIM)} = \beta_i \beta_j \sigma_M^2 + \mathbf{1}_{\{i=j\}}\sigma_{\varepsilon_i}^2.
$$

## 4a — Indeks: **WIG20**.

In [ ]:
# --- 4a/4b: estymacja alfa, beta, sigma2_eps dla każdej akcji ---
R_M = log_ret[INDEX].values                       # dzienne log-zwroty WIG20
mu_M  = R_M.mean() * TRADING_DAYS
sig_M = R_M.std()  * np.sqrt(TRADING_DAYS)
var_M = sig_M ** 2

print(f'Indeks WIG20:  E(R_M) = {mu_M:.4f},  σ_M = {sig_M:.4f}')

sim_params = []
for name in names5:
    y = R5[name].values
    x = R_M
    # OLS: y = alpha + beta*x + eps
    cov_xy = np.cov(x, y, ddof=1)
    beta_d  = cov_xy[0, 1] / cov_xy[0, 0]
    alpha_d = y.mean() - beta_d * x.mean()
    resid   = y - (alpha_d + beta_d*x)
    sig2_eps_d = resid.var(ddof=2)
    # Annualizacja: alpha_year = alpha_day * 252; beta bez zmian; var_eps_year = var_eps_day * 252
    sim_params.append({
        'Akcja':   name,
        'alpha (rok)':    alpha_d * TRADING_DAYS,
        'beta':           beta_d,
        'sigma2_eps (rok)': sig2_eps_d * TRADING_DAYS,
        'R^2':     1 - resid.var(ddof=2) / y.var(ddof=1),
    })
sim_df = pd.DataFrame(sim_params).set_index('Akcja').round(5)
display(sim_df)

alphas = sim_df['alpha (rok)'].values
betas  = sim_df['beta'].values
s2_eps = sim_df['sigma2_eps (rok)'].values

In [ ]:
# --- 4c: macierz kowariancji z modelu jednoskaźnikowego ---
Sigma_SIM = np.outer(betas, betas) * var_M + np.diag(s2_eps)

print('Macierz kowariancji (model jednoskaźnikowy, roczna):')
display(pd.DataFrame(Sigma_SIM, index=names5, columns=names5).round(5))

print('\nMacierz kowariancji (próbkowa, roczna):')
display(pd.DataFrame(Sigma5, index=names5, columns=names5).round(5))

# Norma Frobeniusa różnicy
diff_norm = np.linalg.norm(Sigma_SIM - Sigma5, 'fro')
print(f'\nNorma Frobeniusa różnicy ||Σ_SIM - Σ_sample||_F = {diff_norm:.5f}')

# Wykres porównawczy macierzy korelacji
def cov_to_corr(S):
    d = np.sqrt(np.diag(S))
    return S / np.outer(d, d)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mat, title in zip(axes, [cov_to_corr(Sigma5), cov_to_corr(Sigma_SIM)],
                          ['Korelacja — próbkowa', 'Korelacja — z modelu jednoskaźnikowego']):
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names5, rotation=30, ha='right'); ax.set_yticklabels(names5)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center',
                    color='black' if abs(mat[i,j])<0.5 else 'white', fontsize=9)
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## 4d — MVP i portfel rynkowy **na bazie $\Sigma_{SIM}$**

Powtarzamy optymalizację z punktu 3, ale używamy macierzy z modelu jednoskaźnikowego.
Wektor oczekiwanych zwrotów liczymy z formuły **SIM**: $\mathbb{E}(R_i) = \alpha_i + \beta_i \mathbb{E}(R_M)$
(można też zostawić $\mu_5$ próbkowe — sprawdzamy oba warianty).

In [ ]:
# --- 4d: MVP i portfel rynkowy z macierzy modelu jednoskaźnikowego ---
mu_SIM = alphas + betas * mu_M  # E(R_i) z SIM
# (alternatywnie: mu_SIM = mu5)

# MVP
res_mvp_S = minimize(port_var, x0=np.ones(n)/n, args=(Sigma_SIM,), method='SLSQP',
                     bounds=bounds, constraints=[cons_sum])
w_mvp_S = res_mvp_S.x
mu_mvp_S  = w_mvp_S @ mu_SIM
sig_mvp_S = np.sqrt(w_mvp_S @ Sigma_SIM @ w_mvp_S)

# Rynkowy
res_mkt_S = minimize(neg_sharpe, x0=np.ones(n)/n, args=(mu_SIM, Sigma_SIM, RF_ANN),
                     method='SLSQP', bounds=bounds, constraints=[cons_sum])
w_mkt_S = res_mkt_S.x
mu_mkt_S  = w_mkt_S @ mu_SIM
sig_mkt_S = np.sqrt(w_mkt_S @ Sigma_SIM @ w_mkt_S)
S_mkt_S   = (mu_mkt_S - RF_ANN) / sig_mkt_S

# Realne (ex post) ryzyko portfeli SIM przy próbkowej Σ
sig_mvp_S_realized = np.sqrt(w_mvp_S @ Sigma5 @ w_mvp_S)
sig_mkt_S_realized = np.sqrt(w_mkt_S @ Sigma5 @ w_mkt_S)

cmp_sim = pd.DataFrame([
    ['MVP — próbkowa Σ', w_mvp5, mu_mvp5,  sig_mvp5,  sig_mvp5],
    ['MVP — SIM Σ',      w_mvp_S, mu_mvp_S, sig_mvp_S, sig_mvp_S_realized],
    ['Rynkowy — próbkowa Σ', w_mkt5, mu_mkt5, sig_mkt5, sig_mkt5],
    ['Rynkowy — SIM Σ',      w_mkt_S, mu_mkt_S, sig_mkt_S, sig_mkt_S_realized],
], columns=['Portfel','Wagi','E(R)','σ (model)','σ realne (Σ_sample)'])
cmp_sim['E(R)']               = cmp_sim['E(R)'].round(4)
cmp_sim['σ (model)']          = cmp_sim['σ (model)'].round(4)
cmp_sim['σ realne (Σ_sample)']= cmp_sim['σ realne (Σ_sample)'].round(4)

print('Wagi MVP (SIM):',     pd.Series(w_mvp_S.round(4), index=names5).to_dict())
print('Wagi rynkowy (SIM):', pd.Series(w_mkt_S.round(4), index=names5).to_dict())
display(cmp_sim[['Portfel','E(R)','σ (model)','σ realne (Σ_sample)']])

In [ ]:
# --- Porównanie graficzne: portfele SIM vs próbkowe ---
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(mc_sig, mc_mu, c='lightgray', s=6, alpha=0.4, label='MC (Σ próbkowa)')
ax.plot(eff_sig, eff_mu, color='red', lw=2, label='Brzeg efektywny (Σ próbkowa)')
for i, name in enumerate(names5):
    ax.scatter(np.sqrt(Sigma5[i,i]), mu5[i], s=80, marker='s', edgecolor='black')
    ax.annotate(name, (np.sqrt(Sigma5[i,i]), mu5[i]),
                textcoords='offset points', xytext=(5,5), fontsize=9)
ax.scatter(sig_mvp5, mu_mvp5, s=180, color='blue',  marker='*', label='MVP (Σ próbkowa)')
ax.scatter(sig_mkt5, mu_mkt5, s=180, color='red',   marker='*', label='Rynkowy (Σ próbkowa)')
ax.scatter(sig_mvp_S_realized, mu_mvp_S, s=180, color='cyan',   marker='X',
           label='MVP (Σ SIM, σ realne)', edgecolor='black')
ax.scatter(sig_mkt_S_realized, mu_mkt_S, s=180, color='orange', marker='X',
           label='Rynkowy (Σ SIM, σ realne)', edgecolor='black')
ax.axhline(RF_ANN, color='black', ls=':', alpha=0.5)
ax.set_xlabel('σ_p (roczne)'); ax.set_ylabel('E(R_p) (roczne)')
ax.set_title('Pkt 4d: Optymalizacja na Σ_SIM vs Σ próbkowej',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

---
# 5. Interpretacja wyników

### Pkt 1 — portfel 2-składnikowy (PKO BP + CD Projekt)
- Dwie akcje z różnych sektorów (banki i tech) mają zauważalnie różną zmienność (CD Projekt
  jest typowo wyraźnie bardziej ryzykowny). Korelacja jest **dodatnia, ale niska**, dzięki
  czemu zbiór możliwych portfeli wyraźnie „wybrzusza się” w lewo (efekt dywersyfikacji).
- **MVP** lokuje większą wagę w mniej zmienną akcję (PKO BP).
- **Portfel rynkowy** (max Sharpe) zwykle leży między MVP a portfelem o najwyższej $E(R)$
  — zależnie od poziomu r_f i Sharpe’a poszczególnych akcji.

### Pkt 2 — VaR i ograniczenie 20 %
- Metoda **wariancji-kowariancji** zakłada normalność stóp zwrotu — dla pojedynczych akcji
  (zwłaszcza CD Projekt) zaniża „grube ogony”, ale dla portfeli z 5 składników jest
  rozsądnym przybliżeniem (efekt centralnego twierdzenia granicznego na poziomie portfela).
- Ograniczenie „strata ≤ 20 % rocznie z prob. 99 %” = warunek
  $\mu_p - 2{,}3263\,\sigma_p \ge -0{,}20$. Spełniają go zwykle portfele bliskie MVP
  i nie spełniają portfele 100 % w jednej spółce o wysokim σ.
- Maksimum $E(R)$ pod ograniczeniem to **punkt styczności** brzegu efektywnego z prostą
  $\mu_p - 2{,}3263\,\sigma_p = -0{,}20$.

### Pkt 3 — dywersyfikacja do 5 akcji
- Po przejściu z 2 do 5 akcji **brzeg efektywny przesuwa się w lewo i w górę**: przy tym
  samym ryzyku można uzyskać większy zwrot, lub przy tym samym zwrocie — mniejsze ryzyko.
- Konkretnie: MVP(5) ma niższe σ niż MVP(2), a portfel rynkowy(5) ma wyższy Sharpe niż
  rynkowy(2) — dywersyfikacja sektorowa **działa**, bo dodajemy aktywa o niskich
  wzajemnych korelacjach.

### Pkt 4 — model jednoskaźnikowy
- Bety estymowane względem WIG20 plasują akcje w intuicyjnej kolejności: KGHM i CD Projekt
  zwykle mają β > 1 (cykliczne, „agresywne”), Dino jako defensywny gracz ma β < 1, PKO BP
  ma β ≈ 1.
- Macierz $\Sigma_{SIM}$ jest **mocno regularyzowana** — wymusza taką samą strukturę
  korelacji (każda para spółek koreluje przez jeden „wspólny czynnik” = WIG20).
- W praktyce $\Sigma_{SIM}$ niedoszacowuje **korelacji branżowych** (np. dwie akcje
  z tego samego sektora skorelowane silniej niż przez wspólny benchmark). Stąd norma
  Frobeniusa różnicy bywa znacząca.
- Portfele zoptymalizowane na $\Sigma_{SIM}$ mają **gładsze wagi** (mniej skrajnych
  alokacji), ale ich realne ryzyko (mierzone $\Sigma$ próbkową) jest zwykle o kilka–kilkanaście
  procent wyższe niż MVP zoptymalizowanego bezpośrednio na $\Sigma$ próbkowej.
- Zaleta SIM: **redukcja liczby parametrów** z $\frac{n(n+1)}{2}$ do $2n+1$ → mniejsze
  ryzyko *overfittingu* — szczególnie przy krótkiej próbie.

### Rekomendacja dla XTB
- Z perspektywy *risk-adjusted return* sensowne jest ulokowanie 1 mln PLN według **portfela
  rynkowego (max Sharpe, 5 akcji)** — patrz alokacja w pkt 3.
- Jeśli zarząd narzuca twardy *risk budget* (VaR 99 % ≤ 20 %), wybieramy **portfel
  brzegowy z pkt 2/3** zaprojektowany pod to ograniczenie.
- Macierz z **modelu jednoskaźnikowego** warto traktować jako *sanity check* — pokazuje,
  ile z ryzyka portfela pochodzi z ekspozycji na WIG20, a ile z ryzyka idiosynkratycznego.
